# Course 1: Geometric Deep Learning and Graphs
## Session 3: How expressive GNNs are

**Goal:** build two different graphs that message passing cannot tell apart, and see which aggregators lose information.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Two graphs: one hexagon, and two triangles
Both have 6 nodes and every node has degree 2, but one is connected and the other is not.

In [ ]:
import numpy as np
from collections import Counter

def cycle(n):
    A = np.zeros((n, n))
    for i in range(n): A[i, (i + 1) % n] = A[(i + 1) % n, i] = 1
    return A

hexagon = cycle(6)
two_triangles = np.block([[cycle(3), np.zeros((3, 3))], [np.zeros((3, 3)), cycle(3)]])
print("triangles in hexagon:", int(np.trace(np.linalg.matrix_power(hexagon, 3)) / 6),
      "| in two triangles:", int(np.trace(np.linalg.matrix_power(two_triangles, 3)) / 6))

### 2. The Weisfeiler-Leman colour refinement test
Each round, a node's new colour is its old colour together with the multiset of its neighbours' colours. If the colour histograms differ, the graphs are different. If they match, the test cannot tell.

In [ ]:
def wl_histogram(A, rounds=4):
    colours = [0] * len(A)
    for _ in range(rounds):
        sigs = [(colours[i], tuple(sorted(colours[j] for j in np.flatnonzero(A[i])))) for i in range(len(A))]
        table = {s: k for k, s in enumerate(sorted(set(sigs)))}
        colours = [table[s] for s in sigs]
    return Counter(colours)

print("hexagon      :", wl_histogram(hexagon))
print("two triangles:", wl_histogram(two_triangles))
assert wl_histogram(hexagon) == wl_histogram(two_triangles)
print("WL cannot tell them apart")

### 3. So a GCN cannot either
With the same starting feature on every node, any message-passing network gives both graphs the same embedding, whatever the weights.

In [ ]:
rng = np.random.default_rng(0)
def gcn_embed(A, Ws):
    A_hat = A + np.eye(len(A)); d = A_hat.sum(1); A_hat = A_hat / np.sqrt(np.outer(d, d))
    H = np.ones((len(A), 1))
    for W in Ws: H = np.tanh(A_hat @ H @ W)
    return H.sum(0)

Ws = [rng.normal(size=(1, 8)), rng.normal(size=(8, 8)), rng.normal(size=(8, 4))]
e1, e2 = gcn_embed(hexagon, Ws), gcn_embed(two_triangles, Ws)
print(e1.round(4)); print(e2.round(4))
assert np.allclose(e1, e2)

### 4. Where sum beats mean and max (the GIN argument)
A centre node with two identical neighbours and a centre with three. Mean and max see the same thing; sum does not.

In [ ]:
neigh_a, neigh_b = np.ones((2, 3)), np.ones((3, 3))
for name, agg in [("mean", lambda H: H.mean(0)), ("max", lambda H: H.max(0)), ("sum", lambda H: H.sum(0))]:
    same = np.allclose(agg(neigh_a), agg(neigh_b))
    print(f"{name:4s}: {'cannot' if same else 'can'} tell 2 neighbours from 3")

### 5. One way out: give the nodes structural features
Counting the triangles each node belongs to, `diag(A³)/2`, separates the two graphs at once.

In [ ]:
tri = lambda A: np.diag(np.linalg.matrix_power(A, 3)) / 2
print("hexagon      :", tri(hexagon))
print("two triangles:", tri(two_triangles))

### TODO
1. Find another pair that WL cannot separate (hint: an 8-cycle and two 4-cycles; then try the 4x4 rook's graph and the Shrikhande graph).
2. Write `gin_embed` with update `MLP((1 + eps) * h + sum of neighbours)` and confirm it also fails on this pair.
3. Which of the fixes from the session (structural features, positional encodings, higher-order WL) would you use for molecules, and why?